# Machine Learning: Regressions

**Objectives:**
- Load and explore a dataset
- Split data into training and test sets
- Train regression models with scikit-learn (Linear, Lasso, Ridge)
- Evaluate model performance using $R^2$ scores
- Understand the purpose of cross-validation (k-fold)

We work through two datasets:  
1. **Diabetes dataset** — a small, pre-normalized dataset to learn the ML workflow  
2. **California Housing dataset** — a larger, real-world dataset to practice sparse regressions

---
## Part 1: Diabetes Dataset — Basic Regression

### Step 1: Import and Explore the Data

We start by loading the **diabetes dataset** from scikit-learn.  
It contains 10 baseline variables (age, sex, BMI, blood pressure, etc.) for 442 patients,  
and a target variable measuring disease progression one year later.

The dataset is returned as a dictionary with keys: `'data'`, `'target'`, `'feature_names'`, `'DESCR'`.

In [ ]:
import sklearn
import sklearn.datasets

dataset = sklearn.datasets.load_diabetes()

# The result is a dictionary:
# 'data'          -> features (X)
# 'target'        -> labels (y)
# 'feature_names' -> names of the features
# 'DESCR'         -> description of the dataset

In [ ]:
print(dataset['DESCR'])

Let's put the data into a **pandas DataFrame** for easier exploration.

In [ ]:
import pandas

df = pandas.DataFrame(dataset['data'], columns=dataset['feature_names'])
df['disease_progression'] = dataset['target']

In [ ]:
df.describe()

> **Interpretation:** Notice that the means of the features are close to zero and the standard deviations are similar across variables. This tells us the data has already been **normalized** (centered and scaled). Normalization is important so that no single feature dominates the regression just because of its scale.

In [ ]:
import seaborn
seaborn.pairplot(df)

---
### Step 2: Split Into Training and Test Sets

**Why do we split?** We want to evaluate our model on data it has *never seen* during training.  
This tells us how well the model **generalizes** to new observations (out-of-sample performance).

We use 70% for training and 30% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Features: one row per observation, one column per feature
print("Features shape:", dataset['data'].shape)

# Target: what we are trying to predict (disease progression)
print("Target shape:", dataset['target'].shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset['data'], dataset['target'],
    test_size=0.3,
    random_state=56  # fixed seed for reproducibility
)

---
### Step 3: Train a Linear Regression Model

**What is linear regression?**  
We fit a model: $\hat{y} = a + b_1 x_1 + b_2 x_2 + \ldots + b_{10} x_{10}$  
where $a$ is the intercept and $b_i$ are the coefficients.

Scikit-learn makes this very simple: create a model object, then call `.fit()`.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()  # create model object
model.fit(X_train, y_train)  # train it on the training data

In [ ]:
# Intercept (a)
print("Intercept:", model.intercept_)

In [ ]:
# Coefficients (b_1, b_2, ..., b_10)
print("Coefficients:", model.coef_)

---
### Step 4: Evaluate the Model

We use the **$R^2$ score** to evaluate model quality.  
$R^2 = 1$ means perfect prediction; $R^2 = 0$ means the model predicts no better than the mean.

We check *both* the training score and the test score to look for **overfitting**.  
If the training score is much higher than the test score, the model memorized the training data rather than learning general patterns.

In [ ]:
print("Test R² score: ", model.score(X_test, y_test))
print("Train R² score:", model.score(X_train, y_train))

> **Interpretation:** The test and training scores are relatively close, so overfitting is not a major concern here. However, the $R^2$ is moderate (~0.5), meaning the linear model captures only about half of the variance in disease progression.

---
### Step 5: Should We Adjust the Test Set Size?

One might wonder: *Does the size of the test set matter?*  Let's try different splits and see how the score changes.


In [ ]:
sizes = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
scores = []

for s in sizes:
    X_train, X_test, y_train, y_test = train_test_split(
        dataset['data'], dataset['target'], test_size=s
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(sizes, scores, marker='o')
plt.xlabel('Test set fraction')
plt.ylabel('R² score')
plt.title('Score vs. test set size')
plt.show()

> **Interpretation:** The score fluctuates depending on the split. This illustrates a key problem: a *single* train/test split gives an unstable estimate of model performance. The solution? **Cross-validation.**

### Step 6: K-Fold Cross-Validation

**Why cross-validation?**  
A single train/test split can give an unstable estimate of model performance because results depend on how the data was randomly divided.  
K-fold cross-validation improves this by repeating the evaluation several times using different parts of the data.

**How it works**

1. The dataset is divided into $k$ equal parts (called folds).  
2. The model is trained $k$ times.  
3. Each time, a different fold is used as the test set, while the remaining $k-1$ folds are used for training.  
4. The $k$ test scores are averaged to obtain a more reliable estimate of performance.

**Train and test proportions**

In K-fold cross-validation, the proportions are fixed by $k$:

Test share: $\frac{1}{k}$  
Train share: $\frac{k-1}{k}$

Examples:

- with $k=3$: Train ≈ 67%, Test ≈ 33%  
- with $k=5$: Train ≈ 80%, Test ≈ 20%

The proportions stay the same in each iteration, but the observations in the test set change.

<div style="display: flex; gap: 40px; align-items: flex-start;">

<div>

**Example (3-fold cross-validation)**

| Fold iteration | Training data | Test data |
|---|---|---|
| Fold 1 | Fold 2 + Fold 3 | Fold 1 |
| Fold 2 | Fold 1 + Fold 3 | Fold 2 |
| Fold 3 | Fold 1 + Fold 2 | Fold 3 |

This means:

- every observation is used for testing exactly once  
- every observation is used for training $k-1$ times  

</div>

<div>

**Observation-level view (3-fold cross-validation)**

| observation | fold block | Fold 1 | Fold 2 | Fold 3 |
|---|---|---|---|---|
| 0 | 0–147 | TEST | TRAIN | TRAIN |
| 1 | 0–147 | TEST | TRAIN | TRAIN |
| ... | ... | ... | ... | ... |
| 148 | 148–295 | TRAIN | TEST | TRAIN |
| ... | ... | ... | ... | ... |
| 441 | 296–441 | TRAIN | TRAIN | TEST |

</div>

</div>

**Computing performance (R² example)**

Suppose the test $R^2$ from each fold is: $R^2_1 = 0.47$, $R^2_2 = 0.49$, $R^2_3 = 0.51$

The cross-validation estimate is: $R^2_{CV} = \frac{0.47 + 0.49 + 0.51}{3} = 0.49$

We report the average test $R^2$ as our estimate of model performance.

In [ ]:
from sklearn.model_selection import KFold

X = dataset['data']
y = dataset['target']
scores = []

kf = KFold(n_splits=3)

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)

print(f"K-Fold scores: {scores}")
print(f"Mean score:   {sum(scores) / len(scores):.2f}")

> **Interpretation:** The k-fold scores give us a more reliable estimate of the model's predictive power. The average across folds is a better summary than any single split.

---
### Step 7: Introduction to Lasso Regression

**Lasso** (Least Absolute Shrinkage and Selection Operator) adds a penalty to the size of the coefficients:  it pushes some coefficients toward zero, effectively performing **variable selection**.

This can be useful when you suspect that not all features are relevant.

Note that for models to be comparable, they must be evaluated on the **same folds**.

In [ ]:
from sklearn.linear_model import Lasso

scores = []

for train_index, test_index in kf.split(X):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model_lasso = Lasso()
    model_lasso.fit(X_train, y_train)

    score = model_lasso.score(X_test, y_test)
    scores.append(score)

print("K-Fold scores:", scores)
print("Mean score:", sum(scores) / len(scores))

> **Interpretation:** The Lasso score on the test set is slightly worse than plain linear regression. This is because the **default regularization parameter** (`alpha=1.0`) may be too strong for this dataset, shrinking useful coefficients too much. Tuning `alpha` could improve performance.

**Optional (advanced): tuning the Lasso penalty**

So far we compared Linear Regression and Lasso using the **same outer K-fold splits**, which ensures a fair comparison.

However, Lasso has a hyperparameter **α (alpha)** controlling the strength of regularization. Using the default value may not be optimal.

To tune α **without leaking information from the test folds**, we use **nested cross-validation**:

• **Inner loop:** choose the best α on the training fold  
• **Outer loop:** evaluate the tuned model on the test fold

This keeps the evaluation fair and usually improves Lasso performance.

In [ ]:

scores = []
best_alphas = []

alpha_grid = [0.001, 0.01, 0.1, 1, 10]

for train_index, test_index in kf.split(X):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    alpha_scores = []

    for alpha in alpha_grid:

        inner_scores = []

        for inner_train_index, inner_test_index in kf.split(X_train):

            X_train_inner = X_train[inner_train_index]
            X_test_inner = X_train[inner_test_index]
            y_train_inner = y_train[inner_train_index]
            y_test_inner = y_train[inner_test_index]

            model_lasso = Lasso(alpha=alpha)
            model_lasso.fit(X_train_inner, y_train_inner)

            inner_scores.append(model_lasso.score(X_test_inner, y_test_inner))

        alpha_scores.append(sum(inner_scores) / len(inner_scores))

    best_alpha = alpha_grid[alpha_scores.index(max(alpha_scores))]
    best_alphas.append(best_alpha)

    model_lasso = Lasso(alpha=best_alpha)
    model_lasso.fit(X_train, y_train)

    score = model_lasso.score(X_test, y_test)
    scores.append(score)

print("Best alpha per fold:", best_alphas)
print("Outer K-Fold scores:", scores)
print("Mean score:", sum(scores) / len(scores))

Interpretation: The Lasso score on the test set is now equvialent linear regression.

**Comparing multiple models with tuning**

We now compare four models (with automatic tuning):
• Linear Regression  
• Lasso 
• Ridge 
• Elastic Net 

The **same outer K-fold splits** are used for all models to ensure a fair comparison.

For the regularized models, the best penalty parameter is selected **inside the training fold** using cross-validation, and the tuned model is then evaluated on the outer test fold.

In [ ]:
scores_linear = []
scores_lasso = []
scores_ridge = []
scores_elastic = []

for train_index, test_index in kf.split(X):

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model_linear = LinearRegression()
    model_linear.fit(X_train, y_train)
    scores_linear.append(model_linear.score(X_test, y_test))

    model_lasso = LassoCV(cv=3)
    model_lasso.fit(X_train, y_train)
    scores_lasso.append(model_lasso.score(X_test, y_test))

    model_ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10])
    model_ridge.fit(X_train, y_train)
    scores_ridge.append(model_ridge.score(X_test, y_test))

    model_elastic = ElasticNetCV(cv=3)
    model_elastic.fit(X_train, y_train)
    scores_elastic.append(model_elastic.score(X_test, y_test))

print(f"Linear mean score: {sum(scores_linear) / len(scores_linear):.2f}")
print(f"Lasso mean score: {sum(scores_lasso) / len(scores_lasso):.2f}")
print(f"Ridge mean score: {sum(scores_ridge) / len(scores_ridge):.2f}")
print(f"Elastic Net mean score: {sum(scores_elastic) / len(scores_elastic):.2f}")

---
## Part 2: California Housing — Sparse Regressions

Now we apply the same workflow to a larger, real-world dataset:  
the **California Housing** dataset (median house values across California districts).

__Import the California Housing Price Dataset from sklearn.__

In [ ]:
from sklearn.datasets import fetch_california_housing
dataset = fetch_california_housing()

__Explore the data (description, correlations, histograms...)__

__Split the dataset into a training set (70%) and a test set (30%).__

__Train  a linear model on the training set (using sklearn). Compute the fitting score on the test set.__ 

__Train a lasso model to predict house prices. Compute the score on the test set.__

__Train a ridge model to predict house prices. Which one is better?__

__(Bonus): For your preferred regression, implement the k-fold evaluation scheme to validate the model.__

__(Bonus): Use k-fold cross-validation to compare Linear Regression, Lasso, Ridge, and Elastic Net. For the regularized models, tune the penalty parameter inside the training folds.__